# NB19 — QAOA and QUBO Optimization of Context Portfolios

**Quantum Brain Research Laboratory — Alejandro Reynoso**


## Purpose

A context window should contain a portfolio of complementary evidence rather than the
highest-scoring path repeated in several forms. We formulate selection as a QUBO:

\[
\max_x \sum_i v_i x_i-\lambda\sum_{i<j}s_{ij}x_ix_j
-\rho\left(\sum_i x_i-k\right)^2,
\]

where \(v_i\) rewards relevance, reliability, novelty and contradiction, \(s_{ij}\)
penalizes redundant paths, and \(k\) is the context budget. We compare exact
enumeration, greedy selection, simulated annealing, and a statevector QAOA prototype.


In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys

IN_COLAB = Path("/content").exists()
LAB_ROOT = Path("/content/Quantum_Brain_Lab") if IN_COLAB else Path.cwd() / "Quantum_Brain_Lab"
LAB_ROOT.mkdir(parents=True, exist_ok=True)
DEPS = LAB_ROOT / "_deps"

required = {
    "networkx": "networkx",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}
missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    DEPS.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--target", str(DEPS), *missing]
    )
    sys.path.insert(0, str(DEPS))

print(f"Laboratory root: {LAB_ROOT}")


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import random
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

SEED = 271828
RNG = random.Random(SEED)
np.random.seed(SEED)

def stable_hash(value: Any) -> str:
    raw = json.dumps(value, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def cosine_words(a: str, b: str) -> float:
    wa = Counter(x.lower().strip(".,:;!?()") for x in a.split())
    wb = Counter(x.lower().strip(".,:;!?()") for x in b.split())
    keys = set(wa) | set(wb)
    if not keys:
        return 0.0
    va = np.array([wa[k] for k in keys], dtype=float)
    vb = np.array([wb[k] for k in keys], dtype=float)
    den = np.linalg.norm(va) * np.linalg.norm(vb)
    return float(va @ vb / den) if den else 0.0

def build_graph() -> nx.MultiDiGraph:
    """Synthetic governed investment-committee vault."""
    g = nx.MultiDiGraph(graph_version="QB-G0", title="Quantum Brain governed vault")
    nodes = {
        "AlphaBank": ("company", "Diversified bank with floating-rate loans, stable deposits, and legacy systems.", 0.00, 0.92),
        "BetaPayments": ("company", "Cloud-native payments platform with rapid growth, thin margins, and rich transaction data.", 0.20, 0.88),
        "ZetaCloud": ("company", "Enterprise cloud provider with recurring revenue and cyber concentration risk.", 0.05, 0.90),
        "GammaRetail": ("company", "Consumer retailer exposed to imported inventory and discretionary demand.", -0.10, 0.87),
        "DeltaLogistics": ("company", "Regional logistics network with pricing power and fuel exposure.", 0.10, 0.86),
        "EtaInsure": ("company", "Insurer benefiting from reinvestment yields while claims rise with inflation.", 0.00, 0.89),
        "RateShock": ("factor", "Policy rates rise sharply, increasing discount rates and funding costs.", -0.70, 0.96),
        "Regulation": ("factor", "Capital, data, conduct, and model-risk requirements tighten.", -0.40, 0.96),
        "CyberShock": ("factor", "A severe cyber event disrupts critical services and customer trust.", -0.80, 0.98),
        "ConsumerSlowdown": ("factor", "Real disposable income and discretionary demand weaken.", -0.70, 0.95),
        "FXShock": ("factor", "The peso depreciates and imported-input costs rise.", -0.60, 0.94),
        "AcquireBeta": ("decision", "AlphaBank considers acquiring BetaPayments.", 0.00, 0.95),
        "ExpandCredit": ("decision", "AlphaBank considers expanding unsecured consumer credit.", 0.00, 0.95),
        "MigrateCloud": ("decision", "The group considers migrating critical workloads to ZetaCloud.", 0.00, 0.95),
        "HedgeFX": ("decision", "GammaRetail considers increasing its foreign-exchange hedge ratio.", 0.00, 0.95),
        "E01": ("evidence", "Payments data can reduce fraud losses and improve cross-selling at AlphaBank.", 0.85, 0.91),
        "E02": ("evidence", "BetaPayments valuation is highly sensitive to higher discount rates.", -0.90, 0.94),
        "E03": ("evidence", "Legacy-control integration creates a material execution risk.", -0.80, 0.93),
        "E04": ("evidence", "Prior acquisitions performed better when product autonomy was preserved.", 0.70, 0.82),
        "E05": ("evidence", "Tighter data regulation increases the fixed cost of payments integration.", -0.75, 0.92),
        "E06": ("evidence", "A bank-payments data estate improves real-time risk detection.", 0.80, 0.89),
        "E07": ("evidence", "Unsecured credit losses rise nonlinearly in consumer slowdowns.", -0.95, 0.97),
        "E08": ("evidence", "Floating-rate assets initially benefit from higher rates.", 0.60, 0.88),
        "E09": ("evidence", "Deposit repricing can later compress the margin benefit.", -0.55, 0.90),
        "E10": ("evidence", "Independent model validation is required before credit expansion.", -0.65, 0.98),
        "E11": ("evidence", "Cloud migration reduces unit costs and improves analytic flexibility.", 0.75, 0.89),
        "E12": ("evidence", "Single-provider concentration can turn a cyber shock into a systemic outage.", -0.95, 0.97),
        "E13": ("evidence", "Workload segmentation contained a prior service disruption.", 0.65, 0.91),
        "E14": ("evidence", "Multi-cloud resilience reduces concentration but raises coordination cost.", 0.25, 0.85),
        "E15": ("evidence", "Layered FX hedges stabilize gross margin.", 0.75, 0.92),
        "E16": ("evidence", "Over-hedging destroys value if currency weakness reverses.", -0.65, 0.88),
        "E17": ("evidence", "FX collateral calls can create a temporary liquidity shock.", -0.55, 0.90),
        "WeakSignalA": ("signal", "A small merchant cohort is moving from cards to account-to-account payments.", 0.45, 0.67),
        "WeakSignalB": ("signal", "New cyber-insurance exclusions may transfer more outage risk to cloud clients.", -0.50, 0.69),
        "OutcomeAutonomy": ("outcome", "Preserved product autonomy accelerated customer migration in a prior deal.", 0.65, 0.94),
        "OutcomeCredit": ("outcome", "A prior downturn generated losses above the linear stress model.", -0.90, 0.96),
        "OutcomeCloud": ("outcome", "Segmentation reduced recovery time during an earlier outage.", 0.70, 0.95),
        "OutcomeHedge": ("outcome", "The hedge protected margin but triggered a collateral call.", 0.10, 0.95),
    }
    for node_id, (kind, text, polarity, reliability) in nodes.items():
        g.add_node(node_id, kind=kind, text=text, polarity=polarity,
                   reliability=reliability, status="authoritative",
                   source=f"source_{1 + len(node_id) % 9:02d}")
    edges = [
        ("AcquireBeta","AlphaBank","concerns"),("AcquireBeta","BetaPayments","concerns"),
        ("ExpandCredit","AlphaBank","concerns"),("MigrateCloud","ZetaCloud","concerns"),
        ("HedgeFX","GammaRetail","concerns"),("AlphaBank","RateShock","exposed_to"),
        ("AlphaBank","Regulation","exposed_to"),("BetaPayments","RateShock","exposed_to"),
        ("BetaPayments","Regulation","exposed_to"),("BetaPayments","ZetaCloud","depends_on"),
        ("ZetaCloud","CyberShock","exposed_to"),("GammaRetail","FXShock","exposed_to"),
        ("GammaRetail","ConsumerSlowdown","exposed_to"),("GammaRetail","DeltaLogistics","depends_on"),
        ("EtaInsure","RateShock","exposed_to"),("EtaInsure","ConsumerSlowdown","exposed_to"),
        ("RateShock","ConsumerSlowdown","causes"),("CyberShock","Regulation","causes"),
        ("FXShock","ConsumerSlowdown","causes"),
        ("E01","AcquireBeta","supports"),("E02","AcquireBeta","contradicts"),
        ("E03","AcquireBeta","contradicts"),("E04","AcquireBeta","supports"),
        ("E05","AcquireBeta","contradicts"),("E06","AcquireBeta","supports"),
        ("WeakSignalA","BetaPayments","informs"),("WeakSignalA","AcquireBeta","supports"),
        ("E04","OutcomeAutonomy","resulted_in"),("OutcomeAutonomy","AcquireBeta","supports"),
        ("E07","ExpandCredit","contradicts"),("E08","ExpandCredit","supports"),
        ("E09","ExpandCredit","contradicts"),("E10","ExpandCredit","contradicts"),
        ("E07","OutcomeCredit","resulted_in"),("OutcomeCredit","ExpandCredit","contradicts"),
        ("E11","MigrateCloud","supports"),("E12","MigrateCloud","contradicts"),
        ("E13","MigrateCloud","supports"),("E14","MigrateCloud","supports"),
        ("WeakSignalB","CyberShock","informs"),("WeakSignalB","MigrateCloud","contradicts"),
        ("E13","OutcomeCloud","resulted_in"),("OutcomeCloud","MigrateCloud","supports"),
        ("E15","HedgeFX","supports"),("E16","HedgeFX","contradicts"),
        ("E17","HedgeFX","contradicts"),("E15","OutcomeHedge","resulted_in"),
        ("OutcomeHedge","HedgeFX","supports"),
        ("E01","E06","corroborates"),("E03","Regulation","informs"),
        ("E05","Regulation","informs"),("E12","CyberShock","informs"),
        ("E17","FXShock","informs"),("E09","RateShock","informs"),
    ]
    for i, (u, v, rel) in enumerate(edges):
        g.add_edge(u, v, rel=rel, weight=0.65 + 0.35 * ((i % 7) / 6))
    return g

QUERIES = [
    {"id":"Q1","text":"Should AlphaBank acquire BetaPayments under higher rates and tighter regulation?",
     "target":"AcquireBeta","seeds":["RateShock","Regulation"],"risk":"high","expected":"caution"},
    {"id":"Q2","text":"Should AlphaBank expand unsecured credit during a consumer slowdown?",
     "target":"ExpandCredit","seeds":["ConsumerSlowdown","RateShock"],"risk":"high","expected":"caution"},
    {"id":"Q3","text":"How should critical workloads migrate to ZetaCloud under cyber risk?",
     "target":"MigrateCloud","seeds":["CyberShock","Regulation"],"risk":"high","expected":"qualified"},
    {"id":"Q4","text":"How should GammaRetail hedge foreign-exchange depreciation risk?",
     "target":"HedgeFX","seeds":["FXShock","ConsumerSlowdown"],"risk":"medium","expected":"qualified"},
]

def simple_graph(g: nx.MultiDiGraph) -> nx.Graph:
    h = nx.Graph()
    h.add_nodes_from(g.nodes(data=True))
    for u, v, d in g.edges(data=True):
        if h.has_edge(u, v):
            h[u][v]["weight"] += float(d.get("weight", 1.0))
        else:
            h.add_edge(u, v, weight=float(d.get("weight", 1.0)), rels={d.get("rel","related")})
    return h

def enumerate_paths(g: nx.MultiDiGraph, query: dict, cutoff: int = 5, cap: int = 80) -> list[list[str]]:
    h = simple_graph(g)
    starts = list(dict.fromkeys(query["seeds"] + [query["target"]]))
    destinations = [n for n, d in h.nodes(data=True)
                    if d.get("kind") in {"evidence","signal","outcome"}]
    paths = []
    for s in starts:
        for t in destinations:
            if s == t:
                continue
            try:
                for p in nx.all_simple_paths(h, s, t, cutoff=cutoff):
                    if query["target"] in p or any(seed in p for seed in query["seeds"]):
                        paths.append(p)
                        if len(paths) >= cap:
                            break
            except nx.NetworkXNoPath:
                pass
            if len(paths) >= cap:
                break
        if len(paths) >= cap:
            break
    unique = []
    seen = set()
    for p in paths:
        key = tuple(p)
        if key not in seen:
            seen.add(key); unique.append(p)
    return unique

def path_features(g: nx.MultiDiGraph, query: dict, path: list[str]) -> dict:
    attrs = [g.nodes[n] for n in path]
    text = " ".join(a.get("text","") for a in attrs)
    evidence = [a for a in attrs if a.get("kind") in {"evidence","outcome","signal"}]
    polarity = float(np.mean([a.get("polarity",0.0) for a in evidence])) if evidence else 0.0
    reliability = float(np.mean([a.get("reliability",0.5) for a in evidence])) if evidence else 0.5
    relevance = cosine_words(query["text"], text)
    kinds = {a.get("kind") for a in attrs}
    novelty = float(sum(a.get("kind") == "signal" for a in attrs) / max(1, len(path)))
    causal = float(any(g.nodes[n].get("kind") == "factor" for n in path)
                   and any(g.nodes[n].get("kind") in {"decision","outcome"} for n in path))
    contradiction = float(polarity < -0.15)
    support = float(polarity > 0.15)
    bridge = float(len(kinds) / 5.0)
    cost = len(path)
    score = (1.7*relevance + 0.9*reliability + 0.45*novelty + 0.40*causal
             + 0.35*bridge + 0.25*contradiction + 0.20*support - 0.06*cost)
    return {
        "relevance": relevance, "reliability": reliability, "novelty": novelty,
        "causal": causal, "contradiction": contradiction, "support": support,
        "bridge": bridge, "cost": cost, "polarity": polarity, "score": score,
    }

def build_catalog(g: nx.MultiDiGraph, queries: list[dict] = QUERIES) -> pd.DataFrame:
    rows = []
    for q in queries:
        for j, path in enumerate(enumerate_paths(g, q)):
            f = path_features(g, q, path)
            rows.append({"query_id":q["id"],"path_id":f"{q['id']}-P{j:03d}",
                         "path":" -> ".join(path),"nodes":path,**f})
    return pd.DataFrame(rows)

def json_graph(g: nx.MultiDiGraph) -> dict:
    return nx.node_link_data(g, edges="edges")

def load_graph(path: Path) -> nx.MultiDiGraph:
    return nx.node_link_graph(json.loads(path.read_text()), edges="edges",
                              directed=True, multigraph=True)

def ensure_baseline() -> tuple[nx.MultiDiGraph, pd.DataFrame]:
    graph_path = LAB_ROOT / "QB_baseline_graph.json"
    catalog_path = LAB_ROOT / "NB17_path_catalog.csv"
    if graph_path.exists() and catalog_path.exists():
        return load_graph(graph_path), pd.read_csv(catalog_path)
    g = build_graph()
    catalog = build_catalog(g)
    graph_path.write_text(json.dumps(json_graph(g), indent=2))
    catalog.assign(nodes=catalog["nodes"].apply(json.dumps)).to_csv(catalog_path, index=False)
    return g, catalog

def normalized_entropy(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > 1e-15]
    return float(-(p*np.log(p)).sum()/np.log(max(2,len(p))))

def jensen_shannon(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p,dtype=float); q=np.asarray(q,dtype=float)
    p=p/p.sum(); q=q/q.sum(); m=0.5*(p+q)
    def kl(a,b):
        mask=a>1e-15
        return float(np.sum(a[mask]*np.log(a[mask]/np.maximum(b[mask],1e-15))))
    return math.sqrt(max(0.0,0.5*kl(p,m)+0.5*kl(q,m)))

def portfolio_metrics(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"paths":0,"mean_score":0,"contradiction_share":0,"support_share":0,
                "novelty":0,"node_coverage":0,"polarity_balance":0}
    nodes = set()
    for value in df["path"]:
        nodes.update(str(value).split(" -> "))
    return {
        "paths":len(df), "mean_score":float(df["score"].mean()),
        "contradiction_share":float(df["contradiction"].mean()),
        "support_share":float(df["support"].mean()),
        "novelty":float(df["novelty"].mean()), "node_coverage":len(nodes),
        "polarity_balance":float(1-abs(df["polarity"].mean())),
    }


In [ ]:
g,catalog=ensure_baseline()
q=QUERIES[0]
frame=catalog[catalog.query_id==q["id"]].nlargest(9,"score").reset_index(drop=True)
n=len(frame); k=4
path_sets=[set(p.split(" -> ")) for p in frame["path"]]
similarity=np.zeros((n,n))
for i in range(n):
    for j in range(n):
        similarity[i,j]=len(path_sets[i]&path_sets[j])/max(1,len(path_sets[i]|path_sets[j]))
value=(frame["score"] + .25*frame["contradiction"] + .20*frame["novelty"]
       + .15*frame["bridge"]).to_numpy()
print(f"Portfolio instance: {n} binary decisions, budget k={k}, search space={2**n:,}")


## 1. Transparent objective and exact benchmark


In [ ]:
redundancy_penalty=.55
budget_penalty=1.45

def objective(bits: np.ndarray) -> float:
    bits=np.asarray(bits,dtype=int)
    base=float(value@bits)
    red=0.0
    for i in range(n):
        for j in range(i+1,n):
            red += similarity[i,j]*bits[i]*bits[j]
    supports=float(frame["support"].to_numpy()@bits)
    challenges=float(frame["contradiction"].to_numpy()@bits)
    balance_reward=.85*min(supports,challenges)
    return (base+balance_reward-redundancy_penalty*red
            -budget_penalty*(bits.sum()-k)**2)

states=np.array(list(itertools.product([0,1],repeat=n)),dtype=int)
costs=np.array([objective(x) for x in states])
feasible=np.where(states.sum(axis=1)==k)[0]
exact_index=feasible[np.argmax(costs[feasible])]
exact_bits=states[exact_index]
exact_score=float(costs[exact_index])
print("Exact optimum:",exact_bits.tolist(),"objective=",round(exact_score,4))
display(frame.loc[np.where(exact_bits==1)[0],["path_id","path","score","polarity"]])


## 2. Classical baselines


In [ ]:
greedy_bits=np.zeros(n,dtype=int)
for _ in range(k):
    options=[]
    for i in range(n):
        if not greedy_bits[i]:
            trial=greedy_bits.copy(); trial[i]=1
            options.append((objective(trial),i))
    greedy_bits[max(options)[1]]=1

def anneal(iterations=3500,temp0=2.2,temp1=.02):
    x=np.zeros(n,dtype=int)
    x[np.random.choice(n,k,replace=False)]=1
    best=x.copy(); best_val=objective(x)
    for step in range(iterations):
        ones=np.where(x==1)[0]; zeros=np.where(x==0)[0]
        trial=x.copy()
        trial[np.random.choice(ones)]=0; trial[np.random.choice(zeros)]=1
        temp=temp0*(temp1/temp0)**(step/max(1,iterations-1))
        delta=objective(trial)-objective(x)
        if delta>=0 or RNG.random()<math.exp(delta/max(temp,1e-9)):
            x=trial
        if objective(x)>best_val:
            best=x.copy(); best_val=objective(x)
    return best,best_val
anneal_bits,anneal_score=anneal()


## 3. Statevector QAOA


In [ ]:
# Basis ordering follows the `states` table above. The phase operator encodes -objective
# as an energy, while the mixer is a product of single-qubit X rotations.
energies=-costs

def mixer_apply(state: np.ndarray,beta: float) -> np.ndarray:
    out=state.copy()
    c=math.cos(beta); s=-1j*math.sin(beta)
    for qubit in range(n):
        stride=2**(n-1-qubit)
        block=2*stride
        new=out.copy()
        for start in range(0,len(out),block):
            for off in range(stride):
                a=start+off; b=a+stride
                va,vb=out[a],out[b]
                new[a]=c*va+s*vb
                new[b]=s*va+c*vb
        out=new
    return out

def qaoa_state(gammas,betas):
    psi=np.ones(2**n,dtype=complex)/math.sqrt(2**n)
    for gamma,beta in zip(gammas,betas):
        psi=psi*np.exp(-1j*gamma*energies)
        psi=mixer_apply(psi,beta)
    return psi/np.linalg.norm(psi)

def expected_objective(params,p=1):
    gammas=params[:p]; betas=params[p:]
    psi=qaoa_state(gammas,betas)
    return float(np.sum(np.abs(psi)**2*costs))

# Reproducible random-grid optimization. This is deliberately inspectable and dependency-free.
best=None
for p in [1,2]:
    trials=220 if p==1 else 420
    for _ in range(trials):
        params=np.r_[np.random.uniform(0,2*np.pi,p),np.random.uniform(0,np.pi,p)]
        val=expected_objective(params,p)
        if best is None or val>best["expectation"]:
            best={"p":p,"params":params,"expectation":val}
psi=qaoa_state(best["params"][:best["p"]],best["params"][best["p"]:])
probs=np.abs(psi)**2
qaoa_index=int(np.argmax(probs))
qaoa_bits=states[qaoa_index]
qaoa_score=objective(qaoa_bits)
feasible_best=feasible[np.argmax(probs[feasible])]
qaoa_feasible_bits=states[feasible_best]
qaoa_feasible_score=objective(qaoa_feasible_bits)
print({"best_depth":best["p"],"expected_objective":round(best["expectation"],4),
       "modal_score":round(qaoa_score,4),"best_feasible_sample_score":round(qaoa_feasible_score,4)})


## 4. Comparative benchmark


In [ ]:
methods={
    "exact":(exact_bits,exact_score),
    "greedy":(greedy_bits,objective(greedy_bits)),
    "simulated_annealing":(anneal_bits,anneal_score),
    "qaoa_feasible_mode":(qaoa_feasible_bits,qaoa_feasible_score),
}
rows=[]
for name,(bits,val) in methods.items():
    chosen=frame.loc[np.where(bits==1)[0]]
    rows.append({"method":name,"objective":val,
                 "approximation_ratio":val/exact_score,
                 "selected_paths":int(bits.sum()),**portfolio_metrics(chosen)})
benchmark=pd.DataFrame(rows)
benchmark.to_csv(LAB_ROOT/"NB19_portfolio_benchmark.csv",index=False)
display(benchmark.round(3))


In [ ]:
top=np.argsort(probs)[-18:][::-1]
sample_df=pd.DataFrame({
    "bitstring":["".join(map(str,states[i])) for i in top],
    "probability":probs[top],"objective":costs[top],
    "selected":states[top].sum(axis=1)
})
sample_df.to_csv(LAB_ROOT/"NB19_qaoa_samples.csv",index=False)
fig,ax=plt.subplots(figsize=(10,4.5))
ax.bar(sample_df.bitstring,sample_df.probability,color="#2E6F95")
ax.tick_params(axis="x",rotation=70); ax.set_ylabel("Measurement probability")
ax.set_title("QAOA state: highest-probability context portfolios")
plt.tight_layout(); plt.savefig(LAB_ROOT/"NB19_qaoa_distribution.png",dpi=180)
plt.show()


In [ ]:
selected_ids={
    name:frame.loc[np.where(bits==1)[0],"path_id"].tolist()
    for name,(bits,_) in methods.items()
}
(LAB_ROOT/"NB19_selected_portfolios.json").write_text(json.dumps(selected_ids,indent=2))
record={
    "object_type":"quantum_navigation_record","query_id":q["id"],
    "algorithm":"statevector_qaoa","backend":"numpy_exact_simulator",
    "qubits":n,"depth":best["p"],"shots":"analytic_statevector",
    "objective":"relevance_reliability_novelty_contradiction_diversity_budget",
    "exact_benchmark":exact_score,"qaoa_feasible_score":qaoa_feasible_score,
    "approximation_ratio":qaoa_feasible_score/exact_score,
    "claim_status":"proof_of_concept_no_hardware_advantage_claim",
    "created_utc":utc_now()
}
(LAB_ROOT/"NB19_manifest.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


## Interpretation

The exact benchmark is indispensable: without it, a high-looking QAOA score says little.
On a small simulator, QAOA is a scientific instrument for testing encodings, objectives
and sampling behaviour—not evidence of computational superiority. The useful design
contribution is the explicit portfolio objective, which forces the system to value
contradiction and diversity rather than only semantic similarity.
